In [ ]:
from transformers import pipeline
import torch
import json

_original_torch_load = torch.load

def _trusted_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _trusted_load

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = pipeline(
#         "text-classification", 
#         model="michellejieli/NSFW_text_classifier", 
#         device=device
#     )

In [ ]:
with open('./training_args.json', 'r') as f:
    training_args = json.load(f)

In [ ]:
training_args

In [ ]:
import pandas as pd
import copy

dialects = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"]
base_path = './results/text_level_toxic_results/final_results_{}.csv'

total_df = pd.read_csv(base_path.format(dialects[0]))[['category', 'standard_prompt', 'dialect_prompt']]
total_df.rename(columns={'dialect_prompt': f'{dialects[0]}_prompt'}, inplace=True)

for dialect in dialects[1:]:
    df = pd.read_csv(base_path.format(dialect))[['standard_prompt', 'dialect_prompt']]
    df.rename(columns={'dialect_prompt': f'{dialect}_prompt'}, inplace=True)
    
    # Merge safely on standard_prompt to prevent row order mismatch
    total_df = pd.merge(total_df, df, on='standard_prompt', how='left')

total_df.rename(columns={'standard_prompt': 'SAE_prompt'}, inplace=True)

In [ ]:
total_df.head(5)

In [ ]:
total_df.to_csv('./original_dataset/toxic_prompts_sae_and_all_dialects.csv')

In [ ]:
benign_df = pd.read_csv('./results/text_level_benign_results/benign_final_results_nsfw_t.csv')

In [ ]:
benign_df = benign_df[['category', 'standard_prompt', 'AAVE_prompt', 'ChcE_prompt', 'CollSgE_prompt', 'IndE_prompt', 'JamE_prompt']]

In [ ]:
benign_df.rename(columns={'standard_prompt': 'SAE_prompt'}, inplace=True)

In [ ]:
benign_df.head(5)

In [ ]:
benign_df.to_csv('./original_dataset/benign_prompts_sae_and_all_dialects.csv')